# Exploratory: plain error correlation as a partner-buying rule

Question: if a site picks the partner **least correlated** with its deployed model (v1-style binary error correlation, measured on all errors), does it do better than picking the **strongest** partner?

Rules compared per anchor: least error correlation; two-stage (partners within 10% of the best marginal quality, then least error correlation); best marginal quality; random (mean over partners); oracle. Outcomes: anchor-relative PDR rescued under the union rule, matched-volume net PDR, incremental false positives.

- **EyePACS:** predictors on the estimation pool (25% eye-level referral); outcomes from `confirmation_pairs_v4.csv` (lock v4 confirmation, already opened).
- **DDR:** predictors on half A, outcomes on half B, and the swap (B to A), at f = 1.31 x half-A prevalence.

Exploratory: all outcome data were previously opened, and this rule was chosen after seeing the manuscript-tables results. CPU, High-RAM, about 15 minutes.

In [1]:
# Cell 1: setup (same machinery as MANUSCRIPT_tables_v2)
from google.colab import drive
drive.mount('/content/drive')
import os, json, itertools, datetime
import numpy as np, pandas as pd
from scipy.stats import rankdata
BASE = '/content/drive/MyDrive/stochastic_orthogonality/data/dr_ensemble_v2'
SPLITS, PREDS, RESULTS = (f'{BASE}/{d}' for d in ('01_splits','03_predictions','04_results'))
ROSTER = ['densenet121_binary','densenet121_5class','efficientnet_b3_binary','efficientnet_b3_5class',
          'resnet50_binary','resnet50_5class','vit_base_binary','vit_base_5class',
          'retfound_binary','retfound_5class','retfound_adversarial']
SEEDS = ['', '_seed1', '_seed2']; W = {4: 500, 3: 200, 2: 15}
ORDERED = [(a, b) for a in ROSTER for b in ROSTER if a != b]
N_ARCH, RNG = 1000, 20260926
def pick(df, c):
    for x in c:
        if x in df.columns: return x
    raise KeyError(f'{c} not in {list(df.columns)}')
def load_preds(ids, tpl):
    P = {}
    for a in ROSTER:
        for s in SEEDS:
            df = pd.read_csv(tpl.format(m=f'{a}{s}'), dtype=str)
            idc = pick(df, ['image_id']); pc = pick(df, ['predicted_probability','probability','prob'])
            v = df.drop_duplicates(subset=idc).set_index(idc)[pc].astype(float).reindex(ids)
            assert v.notna().all(); P[f'{a}{s}'] = v.to_numpy(np.float64)
    return P
def refer(score, frac, mask=None):
    n = len(score); idx = np.arange(n) if mask is None else np.where(mask)[0]
    k = int(round(frac * len(idx))); m = np.zeros(n, bool); m[idx[np.argsort(-score[idx], kind='stable')[:k]]] = True; return m
def pct(score, mask=None):
    n = len(score); out = np.zeros(n); idx = np.arange(n) if mask is None else np.where(mask)[0]
    out[idx] = rankdata(score[idx]) / len(idx); return out
def combos(a, b): return [(f'{a}{sa}', f'{b}{sb}') for sa in SEEDS for sb in SEEDS]

def predictors(P, grade, frac, mask=None):
    n = len(grade); keep = np.ones(n, bool) if mask is None else mask; referable = grade >= 2
    Wm = np.zeros(n); [Wm.__setitem__(grade == g, w) for g, w in W.items()]
    ref = {k: refer(v, frac, mask) for k, v in P.items()}
    marg = {a: float(np.mean([Wm[referable & ~ref[f'{a}{s}'] & keep].sum() for s in SEEDS])) for a in ROSTER}
    rho = {}
    for a, b in itertools.combinations(ROSTER, 2):
        r = [np.corrcoef((referable != ref[ka])[keep].astype(float), (referable != ref[kb])[keep].astype(float))[0, 1] for ka, kb in combos(a, b)]
        rho[(a, b)] = rho[(b, a)] = float(np.nanmean(r))
    return rho, marg
def outcomes(P, grade, frac, mask):
    referable = grade >= 2; pdr = grade == 4
    ref = {k: refer(v, frac, mask) for k, v in P.items()}; rk = {k: pct(v, mask) for k, v in P.items()}; O = {}
    for a, b in ORDERED:
        acc = {'rescued': [], 'mv_net': [], 'incr_fp': []}
        for ka, kb in combos(a, b):
            ra, rb = ref[ka], ref[kb]; e = refer(np.maximum(rk[ka], rk[kb]), frac, mask)
            acc['rescued'].append((pdr & ~ra & rb & mask).sum()); acc['incr_fp'].append((rb & ~ra & ~referable & mask).sum())
            acc['mv_net'].append((pdr & ~ra & mask).sum() - (pdr & ~e & mask).sum())
        O[(a, b)] = {k: float(np.mean(v)) for k, v in acc.items()}
    return O
print('ready')

Mounted at /content/drive
ready


In [2]:
# Cell 2: load both sites and compute predictors and outcomes
ec = pd.read_csv(f'{SPLITS}/eyepacs_split_kaggletest_estconf_v2.csv', dtype={'image_id': str})
pool = ec[pick(ec, ['pool','split','set','assignment','estconf'])].astype(str).str.lower()
est = ec[pool.str.startswith('est')].sort_values('image_id').reset_index(drop=True)
PE = load_preds(est.image_id.to_numpy(), PREDS + '/eyepacs_predictions_kaggletest_{m}_v2.csv')
RHO_E, M_E = predictors(PE, est[pick(est, ['level','grade'])].to_numpy().astype(int), 0.25)
cp = pd.read_csv(f'{RESULTS}/confirmation_pairs_v4.csv')
O_E = {(r.anchor, r.partner): {'rescued': float(r.conf_rescued), 'mv_net': float(r.conf_mv_net), 'incr_fp': float(r.conf_incr_fp)} for _, r in cp.iterrows()}

dd = pd.read_csv(f'{SPLITS}/ddr_split_intraset_v2.csv', dtype={'image_id': str}).sort_values('image_id').reset_index(drop=True)
PD = load_preds(dd.image_id.to_numpy(), PREDS + '/ddr_predictions_{m}_v2.csv')
gD = dd.grade.to_numpy().astype(int); A = (dd.half == 'A').to_numpy(); B = ~A
F = 1.31 * float((gD[A] >= 2).mean())
RHO_A, M_A = predictors(PD, gD, F, A); O_B = outcomes(PD, gD, F, B)
RHO_B, M_B = predictors(PD, gD, F, B); O_A = outcomes(PD, gD, F, A)
print(f'EyePACS estimation {len(est):,} eyes; DDR f = {F:.3f}')

EyePACS estimation 35,718 eyes; DDR f = 0.555


In [3]:
# Cell 3: rules and paired comparisons
def run(rho, marg, O, roster=ROSTER):
    per = {}
    for a in roster:
        c = [x for x in roster if x != a]; mmin = min(marg[x] for x in c); tier = [x for x in c if marg[x] <= 1.10 * mmin]
        picks = {'least_error_correlation': min(c, key=lambda x: rho[(a, x)]),
                 'two_stage_error_correlation': min(tier, key=lambda x: (rho[(a, x)], marg[x])),
                 'best_marginal': min(c, key=lambda x: marg[x])}
        per[a] = {r: O[(a, x)] for r, x in picks.items()}
        per[a]['random'] = {k: float(np.mean([O[(a, x)][k] for x in c])) for k in ('rescued','mv_net','incr_fp')}
        per[a]['oracle'] = {'rescued': float(max(O[(a, x)]['rescued'] for x in c))}
        per[a]['picks'] = picks
    means = {r: {k: round(float(np.mean([per[a][r][k] for a in roster])), 3) for k in ('rescued','mv_net','incr_fp') if k in per[roster[0]][r]}
             for r in ('least_error_correlation','two_stage_error_correlation','best_marginal','random','oracle')}
    rng = np.random.default_rng(RNG); diffs = {}
    for r in ('least_error_correlation', 'two_stage_error_correlation'):
        for k in ('rescued', 'mv_net', 'incr_fp'):
            d = np.array([per[a][r][k] - per[a]['best_marginal'][k] for a in roster])
            bs = [rng.choice(d, len(d), replace=True).mean() for _ in range(N_ARCH)]
            diffs[f'{r}_minus_best_marginal__{k}'] = {'mean': round(float(d.mean()), 3),
                'arch_95': [round(float(np.percentile(bs, 2.5)), 3), round(float(np.percentile(bs, 97.5)), 3)],
                'anchors_better': int((d > 0).sum()) if k != 'incr_fp' else int((d < 0).sum())}
    gain = {r: round((means[r]['rescued'] - means['random']['rescued']) / (means['oracle']['rescued'] - means['random']['rescued']), 3)
            for r in ('least_error_correlation','two_stage_error_correlation','best_marginal')}
    return {'rule_means': means, 'fraction_of_oracle_gain_rescued': gain, 'paired_vs_best_marginal': diffs,
            'picks': {a: per[a]['picks'] for a in roster}}
RES = {'EyePACS_est_to_conf': run(RHO_E, M_E, O_E), 'DDR_A_to_B': run(RHO_A, M_A, O_B), 'DDR_B_to_A': run(RHO_B, M_B, O_A)}
for site, r in RES.items():
    print(f'\n=== {site} ==='); print(pd.DataFrame(r['rule_means']).T.to_string())
    print('fraction of oracle gain:', r['fraction_of_oracle_gain_rescued'])
    for k, v in r['paired_vs_best_marginal'].items(): print(f'  {k:60s} {v}')


=== EyePACS_est_to_conf ===
                             rescued  mv_net   incr_fp
least_error_correlation       14.576   9.626  1158.606
two_stage_error_correlation   14.697   9.445  1117.970
best_marginal                 13.848  10.949   927.394
random                        12.754   6.640  1074.855
oracle                        16.031     NaN       NaN
fraction of oracle gain: {'least_error_correlation': 0.556, 'two_stage_error_correlation': 0.593, 'best_marginal': 0.334}
  least_error_correlation_minus_best_marginal__rescued         {'mean': 0.727, 'arch_95': [-0.232, 1.707], 'anchors_better': 5}
  least_error_correlation_minus_best_marginal__mv_net          {'mean': -1.323, 'arch_95': [-2.848, -0.101], 'anchors_better': 2}
  least_error_correlation_minus_best_marginal__incr_fp         {'mean': 231.212, 'arch_95': [40.34, 437.202], 'anchors_better': 0}
  two_stage_error_correlation_minus_best_marginal__rescued     {'mean': 0.849, 'arch_95': [-0.06, 1.748], 'anchors_better': 5}
  t

In [4]:
# Cell 4: write
out = {'generated_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
       'status': 'exploratory; rule chosen after manuscript_tables_v2; all outcome data previously opened under locks v4 and v5.1',
       'predictor': 'Pearson correlation of binary misclassification vectors at the site operating point, mean over 9 seed combinations',
       'results': RES}
with open(f'{RESULTS}/buying_rule_error_correlation_v2.json', 'w') as f: json.dump(out, f, indent=2)
print('Wrote 04_results/buying_rule_error_correlation_v2.json')

Wrote 04_results/buying_rule_error_correlation_v2.json
